# Minimal RAG in a Notebook

Part of [RAG Lab](https://github.com/sitharaj88/rag-lab). This builds the smallest useful Retrieval-Augmented Generation system: embed a few documents, retrieve the most relevant one for a question, and (optionally) ask a local LLM to answer using only that context.

It runs on **Colab with no API key** — the retrieval half uses `sentence-transformers`. The generation cell at the end is optional and needs a local [Ollama](https://ollama.com) runtime.

> Companion tutorial: *Tutorials → Minimal RAG* on the RAG Lab site.

## 1. Install dependencies

In [ ]:
%pip install -q sentence-transformers numpy

## 2. Our tiny knowledge base

In a real system these would be chunks loaded from your files. Here we just hard-code a few facts.

In [ ]:
documents = [
    "RAG stands for Retrieval-Augmented Generation. It gives a language model access to your own documents at answer time.",
    "Embeddings turn text into lists of numbers so that similar meanings end up close together.",
    "Chunk overlap means adjacent chunks share some text, so an idea split across a boundary is still retrievable.",
    "Ollama lets you run open-source LLMs like Llama 3.2 locally, with no API key.",
    "A vector database stores embeddings and finds the nearest ones to a query very quickly.",
]

## 3. Embed the documents

We load a small, fast embedding model and turn each document into a vector. `normalize_embeddings=True` lets us compare them with a simple dot product.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(documents, normalize_embeddings=True)
print("Each document is now a vector of length", doc_embeddings.shape[1])

## 4. Retrieve the most relevant documents

We embed the question the same way, then rank documents by cosine similarity (a dot product, since the vectors are normalized). The highest scores are the best matches.

In [ ]:
def retrieve(question, top_k=2):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    scores = doc_embeddings @ q          # cosine similarity for each document
    top = np.argsort(scores)[::-1][:top_k]
    return [(documents[i], float(scores[i])) for i in top]

question = "How can I run an LLM without paying for an API?"
for text, score in retrieve(question):
    print(f"{score:.3f}  {text}")

## 5. Build the augmented prompt

This is the *augmented* part of RAG: we stuff the retrieved context into the prompt and instruct the model to answer only from it.

In [ ]:
def build_prompt(question, retrieved):
    context = "\n".join(f"- {text}" for text, _ in retrieved)
    return (
        "Use ONLY the context below to answer. If it doesn't contain the answer, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )

print(build_prompt(question, retrieve(question)))

## 6. (Optional) Generate an answer with a local LLM

This cell needs a local **Ollama** runtime (`ollama serve` + `ollama pull llama3.2`). It won't work on a plain Colab runtime — skip it if you're just exploring retrieval.

In [ ]:
try:
    import ollama
    retrieved = retrieve(question)
    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": build_prompt(question, retrieved)}],
        options={"temperature": 0.1},
    )
    print(response["message"]["content"])
except Exception as e:
    print("Skipping generation (need a local Ollama runtime):", e)

## You built RAG! 🎉

Five documents, one embedding model, a dot product, and a prompt. Everything else in RAG Lab — vector databases, chunking, reranking, evaluation — is about doing each of these steps *better* and *at scale*.

**Next:** try the `02-rag-evaluation.ipynb` notebook to measure how good your retrieval actually is.